# 09 - AutoGluon Tabular Benchmark

## Objective

Evaluate AutoGluon Tabular on the validated Store Sales tabular forecasting pipeline.

The goal is to understand:

- which models AutoGluon trains internally
- which model or ensemble performs best
- how AutoGluon performs across walk-forward validation folds
- how stable AutoGluon is over time
- whether AutoML provides practical value compared with the previously validated LightGBM reference

This notebook trains only AutoGluon.

The manual LightGBM reference is not retrained here because it was already validated in previous notebooks.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import sys
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.pyfunc
from autogluon.tabular import TabularPredictor

sys.path.append(str(Path().resolve().parent))

from src.features import add_baseline_features, add_advanced_features, encode_family
from src.metrics import evaluate_regression
from src.split import build_walk_forward_folds
from src.automl import evaluate_autogluon

## Setup

Use the full-data setup and the same validated feature engineering pipeline from previous notebooks.

This benchmark uses:

- full training history
- 2 recent walk-forward folds
- 28-day validation windows
- 600 seconds AutoGluon time budget per fold
- MLflow tracking

In [4]:
SEED = 42

DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../artifacts/nb9_autogluon_tabular")
AUTOGLUON_DIR = ARTIFACTS_DIR / "autogluon_models"

MLFLOW_TRACKING_URI = "file:../mlruns"
MLFLOW_EXPERIMENT_NAME = "store_sales_forecasting"

N_FOLDS = 2
VAL_SIZE = 28

AUTOGLUON_TIME_LIMIT = 600
AUTOGLUON_PRESETS = "medium_quality"

METRICS_AG_PATH = ARTIFACTS_DIR / "nb9_autogluon_fold_metrics.csv"
SUMMARY_AG_PATH = ARTIFACTS_DIR / "nb9_autogluon_summary.csv"
LEADERBOARD_AG_PATH = ARTIFACTS_DIR / "nb9_autogluon_leaderboard.csv"
FOLDS_PATH = ARTIFACTS_DIR / "nb9_validation_folds.csv"

FORCE_AUTOML_RETRAIN = False

# Reference metrics from NB5 full-data LightGBM validation
NB5_FULL_RMSLE_MEAN = 0.599880
NB5_FULL_MAE_MEAN = 68.447697
NB5_FULL_RMSE_MEAN = 248.706697
NB5_FULL_R2_MEAN = 0.965119

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
AUTOGLUON_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print("Data dir:", DATA_DIR.resolve())
print("Artifacts dir:", ARTIFACTS_DIR.resolve())
print("AutoGluon time limit per fold:", AUTOGLUON_TIME_LIMIT)
print("Number of folds:", N_FOLDS)

Setup OK
Data dir: /home/donatocorbacio/projects/store-sales-project/data
Artifacts dir: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular
AutoGluon time limit per fold: 600
Number of folds: 2


## MLflow setup

Track the AutoGluon benchmark in the same MLflow experiment used by the previous notebooks.

This keeps the AutoML benchmark comparable with the validated LightGBM reference.

In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Active MLflow experiment:", MLFLOW_EXPERIMENT_NAME)

Active MLflow experiment: store_sales_forecasting


## Load data

Load the Store Sales train and test datasets.

In [6]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train range:", train["date"].min(), "->", train["date"].max())
print("Test range:", test["date"].min(), "->", test["date"].max())

display(train.head())

Train shape: (3000888, 6)
Test shape: (28512, 5)
Train range: 2013-01-01 00:00:00 -> 2017-08-15 00:00:00
Test range: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


## Base preprocessing

Sort the dataset by store, family and date.

This is required before creating lag and rolling features.

In [7]:
train = train.sort_values(["store_nbr", "family", "date"]).copy()
test = test.sort_values(["store_nbr", "family", "date"]).copy()

for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

display(train.head())

,id,date,store_nbr,family,sales,onpromotion,year,month,day,dayofweek,weekofyear,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,2013,1,1,1,1,0
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2013,1,2,2,1,0
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,2013,1,3,3,1,0
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,2013,1,4,4,1,0
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,2013,1,5,5,1,1


## Rebuild validated feature pipeline

Reuse the advanced feature pipeline validated in the previous notebooks.

No new features are introduced in this notebook.

In [8]:
advanced_train = add_baseline_features(train)
advanced_train = add_advanced_features(advanced_train)
advanced_train = advanced_train.dropna().copy()

advanced_train, advanced_test, family_mapping = encode_family(advanced_train, test)

advanced_features = [
    "store_nbr",
    "family",
    "onpromotion",
    "year",
    "month",
    "day",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_14",
    "trend_1_7",
    "promo_last_7",
]

print("Advanced train shape:", advanced_train.shape)
print("Advanced date range:", advanced_train["date"].min(), "->", advanced_train["date"].max())
print("Number of features:", len(advanced_features))

Advanced train shape: (2975940, 19)
Advanced date range: 2013-01-15 00:00:00 -> 2017-08-15 00:00:00
Number of features: 16


## Build walk-forward validation folds

Use the same recent anchored walk-forward validation logic used in previous full-data experiments.

Each fold:

- trains on all data available up to a cutoff date
- validates on the next 28 days
- moves forward toward the most recent period

In [9]:
folds = build_walk_forward_folds(
    df=advanced_train,
    n_folds=N_FOLDS,
    val_size=VAL_SIZE,
)

folds_df = pd.DataFrame(folds)
display(folds_df)

for fold_info in folds:
    print(
        f"Fold {fold_info['fold']} | "
        f"train <= {pd.Timestamp(fold_info['train_end']).date()} | "
        f"val: {pd.Timestamp(fold_info['val_start']).date()} -> "
        f"{pd.Timestamp(fold_info['val_end']).date()}"
    )

,fold,train_end,val_start,val_end
0,1,2017-06-20,2017-06-21,2017-07-18
1,2,2017-07-18,2017-07-19,2017-08-15


Fold 1 | train <= 2017-06-20 | val: 2017-06-21 -> 2017-07-18
Fold 2 | train <= 2017-07-18 | val: 2017-07-19 -> 2017-08-15


## Run AutoGluon benchmark with MLflow

This cell runs the full AutoGluon benchmark.

It logs:

- fold metrics
- average metrics
- best model per fold
- AutoGluon leaderboard
- validation folds
- runtime information

In [10]:
with mlflow.start_run(run_name="benchmark_autogluon_tabular_full_cv"):

    mlflow.set_tags({
        "project": "store_sales_forecasting",
        "notebook": "nb9",
        "experiment_type": "automl_benchmark",
        "benchmark_type": "autogluon_tabular",
        "validation_strategy": "walk_forward",
        "dataset_scope": "full_data",
        "feature_set": "advanced_v1",
        "production_candidate": "false",
        "model_registry": "false",
    })

    mlflow.log_params({
        "n_folds": N_FOLDS,
        "val_size_days": VAL_SIZE,
        "n_features": len(advanced_features),
        "feature_set": "advanced_v1",
        "target": "sales",
        "autogluon_model": "TabularPredictor",
        "autogluon_presets": AUTOGLUON_PRESETS,
        "autogluon_time_limit_sec": AUTOGLUON_TIME_LIMIT,
        "nb5_reference_rmsle": NB5_FULL_RMSLE_MEAN,
        "nb5_reference_mae": NB5_FULL_MAE_MEAN,
        "nb5_reference_rmse": NB5_FULL_RMSE_MEAN,
        "nb5_reference_r2": NB5_FULL_R2_MEAN,
    })

    if (
        METRICS_AG_PATH.exists()
        and SUMMARY_AG_PATH.exists()
        and LEADERBOARD_AG_PATH.exists()
        and not FORCE_AUTOML_RETRAIN
    ):
        print("Loading cached AutoGluon benchmark artifacts...")

        metrics_ag = pd.read_csv(
            METRICS_AG_PATH,
            parse_dates=["train_end", "val_start", "val_end"],
        )

        comparison_summary = pd.read_csv(SUMMARY_AG_PATH)
        ag_leaderboard_all = pd.read_csv(LEADERBOARD_AG_PATH)

        ag_leaderboards = [
            fold_df.copy()
            for _, fold_df in ag_leaderboard_all.groupby("fold")
        ]

    else:
        print("Training AutoGluon benchmark...")

        metrics_ag, ag_leaderboards = evaluate_autogluon(
            df=advanced_train,
            features=advanced_features,
            folds=folds,
            model_dir=AUTOGLUON_DIR,
            time_limit=AUTOGLUON_TIME_LIMIT,
            presets=AUTOGLUON_PRESETS,
        )

        comparison_summary = pd.DataFrame([{
            "model": "autogluon_tabular",
            "rmsle_mean": metrics_ag["rmsle"].mean(),
            "rmsle_std": metrics_ag["rmsle"].std(),
            "mae_mean": metrics_ag["mae"].mean(),
            "mae_std": metrics_ag["mae"].std(),
            "rmse_mean": metrics_ag["rmse"].mean(),
            "r2_mean": metrics_ag["r2"].mean(),
            "train_time_mean_sec": metrics_ag["train_time_sec"].mean(),
            "inference_time_mean_sec": metrics_ag["inference_time_sec"].mean(),
            "nb5_reference_rmsle": NB5_FULL_RMSLE_MEAN,
            "nb5_reference_mae": NB5_FULL_MAE_MEAN,
            "nb5_reference_rmse": NB5_FULL_RMSE_MEAN,
            "nb5_reference_r2": NB5_FULL_R2_MEAN,
        }])

        metrics_ag.to_csv(METRICS_AG_PATH, index=False)
        comparison_summary.to_csv(SUMMARY_AG_PATH, index=False)
        folds_df.to_csv(FOLDS_PATH, index=False)

        if len(ag_leaderboards) > 0:
            ag_leaderboard_all = pd.concat(ag_leaderboards, ignore_index=True)
            ag_leaderboard_all.to_csv(LEADERBOARD_AG_PATH, index=False)

    display(metrics_ag)
    display(comparison_summary)

    for _, row in metrics_ag.iterrows():
        fold = int(row["fold"])

        mlflow.log_metric(f"fold_{fold}_rmsle", row["rmsle"])
        mlflow.log_metric(f"fold_{fold}_mae", row["mae"])
        mlflow.log_metric(f"fold_{fold}_rmse", row["rmse"])
        mlflow.log_metric(f"fold_{fold}_r2", row["r2"])
        mlflow.log_metric(f"fold_{fold}_train_time_sec", row["train_time_sec"])
        mlflow.log_metric(
            f"fold_{fold}_inference_time_sec",
            row["inference_time_sec"],
        )
        mlflow.log_param(f"fold_{fold}_best_model", row["best_model"])

    summary_row = comparison_summary.iloc[0]

    mlflow.log_metric("rmsle_mean", summary_row["rmsle_mean"])
    mlflow.log_metric("rmsle_std", summary_row["rmsle_std"])
    mlflow.log_metric("mae_mean", summary_row["mae_mean"])
    mlflow.log_metric("mae_std", summary_row["mae_std"])
    mlflow.log_metric("rmse_mean", summary_row["rmse_mean"])
    mlflow.log_metric("r2_mean", summary_row["r2_mean"])
    mlflow.log_metric("train_time_mean_sec", summary_row["train_time_mean_sec"])
    mlflow.log_metric(
        "inference_time_mean_sec",
        summary_row["inference_time_mean_sec"],
    )

    mlflow.log_artifact(str(METRICS_AG_PATH))
    mlflow.log_artifact(str(SUMMARY_AG_PATH))
    mlflow.log_artifact(str(FOLDS_PATH))

    if LEADERBOARD_AG_PATH.exists():
        mlflow.log_artifact(str(LEADERBOARD_AG_PATH))

    print("MLflow run completed.")

Loading cached AutoGluon benchmark artifacts...


,fold,train_end,val_start,val_end,n_train,n_val,train_time_sec,inference_time_sec,autogluon_time_limit,autogluon_presets,best_model,best_model_score_val,best_model_fit_time,best_model_pred_time_val,rmsle,mae,rmse,r2
0,1,2017-06-20,2017-06-21,2017-07-18,2876148,49896,603.996969,14.644537,600,medium_quality,WeightedEnsemble_L2,-232.503638,586.261508,8.113820,0.476525,53.084927,203.580231,0.976389
1,2,2017-07-18,2017-07-19,2017-08-15,2926044,49896,601.573003,14.101837,600,medium_quality,WeightedEnsemble_L2,-156.877041,587.524698,7.493559,0.475665,58.002139,198.500102,0.975591


,model,rmsle_mean,rmsle_std,mae_mean,mae_std,rmse_mean,r2_mean,train_time_mean_sec,inference_time_mean_sec,nb5_reference_rmsle,nb5_reference_mae,nb5_reference_rmse,nb5_reference_r2
0,autogluon_tabular,0.476095,0.000608,55.543533,3.476993,201.040167,0.97599,602.784986,14.373187,0.59988,68.447697,248.706697,0.965119


MLflow run completed.


## 10 — Final Conclusions

This notebook evaluated AutoGluon Tabular as an AutoML benchmark on the validated Store Sales forecasting pipeline.

The benchmark reused the same full-data feature engineering pipeline and walk-forward validation logic used in the previous LightGBM experiments, allowing a fair comparison between manual modeling and AutoML.

### Key findings

- AutoGluon consistently selected `WeightedEnsemble_L2` as the best model across both recent validation folds.
- The ensemble combined LightGBM-based models (`LightGBM` and `LightGBMXT`) and outperformed the previous manual LightGBM reference.
- AutoGluon achieved stronger average validation performance:
  - RMSLE: 0.4761 vs 0.5999
  - MAE: 55.54 vs 68.45
  - RMSE: 201.04 vs 248.71
  - R²: 0.9760 vs 0.9651

### Practical interpretation

AutoGluon proved to be a strong AutoML benchmark for this tabular forecasting problem.

Its main value was not discovering radically different model families, but automatically building a stronger LightGBM-based ensemble with limited engineering effort.

However, this improvement came with trade-offs:

- higher computational cost (~10 minutes per fold)
- reduced transparency compared with a manually controlled pipeline
- lower control over model behavior and training strategy

### Final takeaway

AutoGluon is a strong benchmarking and experimentation tool for tabular forecasting.

For this project, it provides clear practical value as a benchmark and candidate model exploration layer.

However, the manually engineered LightGBM pipeline remains preferable when interpretability, control and deployment simplicity are stronger priorities.

| Aspect                | Manual LightGBM | AutoGluon |
| --------------------- | --------------- | --------- |
| Performance           | Good            | Better    |
| Training Cost         | Lower           | Higher    |
| Interpretability      | Higher          | Lower     |
| Production Simplicity | Easier          | Harder    |


In [11]:
class AutoGluonWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.predictor = TabularPredictor.load(context.artifacts["predictor_path"])

    def predict(self, context, model_input):
        preds = self.predictor.predict(model_input)
        preds = np.clip(preds, 0, None)
        return preds


NB9_RUN_ID = "167cd025e5e44acc80a4f6eac154518f"

best_fold_model_path = str(AUTOGLUON_DIR / "fold_2")

input_example = advanced_train[advanced_features].head(5)

with mlflow.start_run(run_id=NB9_RUN_ID):

    mlflow.set_tags({
        "model_artifact_logged": "true",
        "model_family": "autogluon_tabular",
        "best_model": "WeightedEnsemble_L2",
        "registry_role": "benchmark_candidate",
        "production_candidate": "false",
    })

    mlflow.pyfunc.log_model(
        artifact_path="autogluon_weighted_ensemble_fold2",
        python_model=AutoGluonWrapper(),
        artifacts={"predictor_path": best_fold_model_path},
        input_example=input_example,
    )

print("AutoGluon model logged inside the existing NB9 MLflow run.")

/home/donatocorbacio/projects/store-sales-project/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/04/30 12:23:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/30 12:23:54 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/04/30 12:23:54 INFO mlflow.pyfunc: Inferring model signature from input example


AutoGluon model logged inside the existing NB9 MLflow run.


In [12]:
# Export NB9 fold-level predictions for NB12

from autogluon.tabular import TabularPredictor

PREDICTIONS_PATH = ARTIFACTS_DIR / "nb9_fold_predictions.csv"

prediction_rows = []

target = "sales"
selected_fold_id = 2

fold_info = [f for f in folds if f["fold"] == selected_fold_id][0]

val_mask = (
    (advanced_train["date"] >= fold_info["val_start"]) &
    (advanced_train["date"] <= fold_info["val_end"])
)

val_df = advanced_train.loc[val_mask].copy()

predictor_path = AUTOGLUON_DIR / f"fold_{selected_fold_id}"
predictor = TabularPredictor.load(str(predictor_path))

val_df["prediction"] = predictor.predict(val_df[advanced_features])
val_df["prediction"] = val_df["prediction"].clip(lower=0)
val_df["fold"] = selected_fold_id

nb9_predictions = val_df[
    ["fold", "date", "store_nbr", "family", "sales", "prediction", "onpromotion"]
].copy()

nb9_predictions.to_csv(PREDICTIONS_PATH, index=False)

print("Saved NB9 predictions to:", PREDICTIONS_PATH.resolve())
display(nb9_predictions.head())

Saved NB9 predictions to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb9_autogluon_tabular/nb9_fold_predictions.csv


,fold,date,store_nbr,family,sales,prediction,onpromotion
2950992,2,2017-07-19,1,0,7.0,5.506095,0
2952774,2,2017-07-20,1,0,4.0,6.113786,0
2954556,2,2017-07-21,1,0,10.0,5.213129,0
2956338,2,2017-07-22,1,0,8.0,4.225158,0
2958120,2,2017-07-23,1,0,0.0,3.790067,0
